# 回归模型验证

本notebook用于训练和评估基于湿度数据预测火灾数量的回归模型。

## 1. 导入必要的库和数据准备

In [2]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os

notebook_dir = os.path.abspath('')
project_root = os.path.dirname(os.path.dirname(notebook_dir))

import data_processor
data_processor.DATA_PATH = os.path.join(project_root, 'data', 'processed')
merged_data, humid_cols = data_processor.prepare_weather_fire_data(
    data_type='count',  # 指定计数预测
    weather_types=['humid']  # 只使用湿度数据
)


合并后的数据信息:
数据形状: (1625, 30)
选择的天气类型: ['humid']



## 2. 数据集划分

In [3]:
from sklearn.model_selection import train_test_split

print("\n正在划分数据集...")
(X_train, y_train), (X_val, y_val), (X_test, y_test) = data_processor.split_dataset(
    merged_data=merged_data,
    feature_columns=humid_cols,  # 只使用湿度特征
    target_column='fire_count'
)

# 数据预处理
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_processed = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)
X_val_processed = pd.DataFrame(
    scaler.transform(X_val),
    columns=X_val.columns,
    index=X_val.index
)
X_test_processed = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("\n数据集形状:")
print(f"训练集: {X_train_processed.shape}")
print(f"验证集: {X_val_processed.shape}")
print(f"测试集: {X_test_processed.shape}")


正在划分数据集...

数据集划分情况:
训练集大小: 975 (60.00%)
验证集大小: 325 (20.00%)
测试集大小: 325 (20.00%)

数据集形状:
训练集: (975, 29)
验证集: (325, 29)
测试集: (325, 29)


## 3. 模型训练与评估

In [1]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

def plot_trends(merged_data, y_pred, model_name, humid_cols, val_indices):
    """
    绘制湿度和火灾数量的趋势关系图
    
    Parameters:
        merged_data: 包含湿度和实际火灾数据的DataFrame
        y_pred: 模型预测的火灾数量
        model_name: 模型名称
        humid_cols: 湿度特征列名列表
        val_indices: 验证集的索引
    """
    # 只使用验证集对应的数据
    val_data = merged_data.loc[val_indices]
    
    # 创建一个包含预测值的DataFrame
    predictions_df = pd.DataFrame({
        'Actual': val_data['fire_count'],
        'Predicted': y_pred
    }, index=val_data.index)
    
    # 1. 时间序列趋势图
    plt.figure(figsize=(15, 6))
    plt.plot(predictions_df.index, predictions_df['Actual'], 
             label='Actual Fire Count', alpha=0.7)
    plt.plot(predictions_df.index, predictions_df['Predicted'], 
             label='Predicted  Fire Count', alpha=0.7)
    plt.title(f'{model_name} - Fire Count Time Series Comparison')
    plt.xlabel('Date')
    plt.ylabel('Fire Count')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    # 2. 湿度-火灾散点图与趋势线
    avg_humid = val_data[humid_cols].mean(axis=1)
    
    plt.figure(figsize=(12, 6))
    plt.scatter(avg_humid, predictions_df['Actual'], 
               alpha=0.5, label='Actual Data', color='blue')
    plt.scatter(avg_humid, predictions_df['Predicted'], 
               alpha=0.5, label='Predicted Data', color='green')
    
    # 添加趋势线
    z1 = np.polyfit(avg_humid, predictions_df['Actual'], 1)
    p1 = np.poly1d(z1)
    plt.plot(avg_humid, p1(avg_humid), "r--", alpha=0.8, label='Actual Trend')
    
    z2 = np.polyfit(avg_humid, predictions_df['Predicted'], 1)
    p2 = np.poly1d(z2)
    plt.plot(avg_humid, p2(avg_humid), "g--", alpha=0.8, label='Predicted Trend')
    
    plt.title(f'{model_name} - Average Humidity vs Fire Count')
    plt.xlabel('Average Humidity')
    plt.ylabel('Fire Count')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.show()

# 定义训练和评估函数
def train_and_evaluate_models(X_train, X_val, y_train, y_val, merged_data, humid_cols):
    """
    训练和评估回归模型
    """
    # 初始化模型
    models = {
        'Linear Regression': LinearRegression(),
        'Random Forest': RandomForestRegressor(
            n_estimators=100,  
            max_depth=10,      
            random_state=42
        )
    }
    
    results = {}
    
    for name, model in models.items():
        print(f"\nTraining {name}...")
        # 训练模型
        model.fit(X_train, y_train)
        
        # 预测
        y_pred = model.predict(X_val)
        
        # 计算评估指标
        mse = mean_squared_error(y_val, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_val, y_pred)
        
        # 打印评估结果
        print(f"{name} Evaluation Results:")
        print(f"Mean Squared Error (MSE): {mse:.4f}")
        print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
        print(f"R-squared (R²): {r2:.4f}")
        
        # 预测值与实际值的散点图
        plt.figure(figsize=(10, 6))
        plt.scatter(y_val, y_pred, alpha=0.5)
        plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--', lw=2)
        plt.title(f'{name} - Actual vs Predicted')
        plt.xlabel('Actual Fire Count')
        plt.ylabel('Predicted Fire Count')
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.show()

        plot_trends(merged_data, y_pred, name, humid_cols, X_val.index)
        
        # 特征重要性图
        if name == 'Random Forest':
            # Random Forest feature importance
            feature_importance = pd.Series(
                model.feature_importances_,
                index=X_train.columns
            ).sort_values(ascending=True)
            
            plt.figure(figsize=(12, 8))
            feature_importance.plot(kind='barh')
            plt.title('Random Forest - Feature Importance')
            plt.xlabel('Importance Score')
            plt.ylabel('Weather Stations')
            plt.grid(True, linestyle='--', alpha=0.3)
            plt.tight_layout()
            plt.show()
        
        if name == 'Linear Regression':
            # Linear Regression feature importance (absolute coefficients)
            linear_importance = pd.Series(
                np.abs(model.coef_),
                index=X_train.columns
            ).sort_values(ascending=True)
            
            plt.figure(figsize=(12, 8))
            linear_importance.plot(kind='barh')
            plt.title('Linear Regression - Feature Importance')
            plt.xlabel('Absolute Coefficient Value')
            plt.ylabel('Weather Stations')
            plt.grid(True, linestyle='--', alpha=0.3)
            plt.tight_layout()
            plt.show()
            
        # Store results
        results[name] = {
            'model': model,
            'mse': mse,
            'rmse': rmse,
            'r2': r2,
            'predictions': y_pred
        }
        
        # Print feature importance rankings
        if name == 'Random Forest':
            print("\nRandom Forest - Top 10 Most Important Features:")
            print(feature_importance.tail(10))
        
        if name == 'Linear Regression':
            print("\nLinear Regression - Top 10 Most Important Features:")
            print(linear_importance.tail(10))
    
    # Compare model performances
    print("\nModel Performance Comparison:")
    performance_comparison = pd.DataFrame({
        'MSE': [results[model]['mse'] for model in models],
        'RMSE': [results[model]['rmse'] for model in models],
        'R²': [results[model]['r2'] for model in models]
    }, index=models.keys())
    print(performance_comparison)
    
    return results

# Configure plot settings for better readability
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10

# Train and evaluate models
print("\nStarting model training and evaluation...")
model_results = train_and_evaluate_models(X_train_processed, X_val_processed, y_train, y_val, merged_data, humid_cols)

# Additional visualization: Model performance comparison
metrics = ['MSE', 'RMSE', 'R²']
model_names = list(model_results.keys())
performance_data = []

for metric in metrics:
    if metric == 'MSE':
        values = [results['mse'] for results in model_results.values()]
    elif metric == 'RMSE':
        values = [results['rmse'] for results in model_results.values()]
    else:  # R²
        values = [results['r2'] for results in model_results.values()]
    performance_data.append(values)

# Plot model performance comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Model Performance Comparison')

for i, (metric, ax) in enumerate(zip(metrics, axes)):
    ax.bar(model_names, performance_data[i])
    ax.set_title(metric)
    ax.set_xticklabels(model_names, rotation=45)
    ax.grid(True, linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()


Starting model training and evaluation...


NameError: name 'X_train_processed' is not defined